# Compare the project datasets
This comparison keeps the Kaggle binary target and UCI three-class target separate. The sources have different populations and feature definitions; side-by-side counts are descriptive, not a pooled estimate of dropout prevalence.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from profiling import profile_dataframe
paths = {'Kaggle synthetic': ROOT/'data/raw/kaggle_synthetic.csv', 'UCI Portuguese': ROOT/'data/raw/uci_portuguese.csv'}
frames = {name: pd.read_csv(path) for name, path in paths.items()}
profiles = {name: profile_dataframe(df) for name, df in frames.items()}
summary = pd.DataFrame({name: {'rows': p['row_count'], 'columns': p['column_count'], 'missing cells': p['missing_cells'], 'duplicate rows': p['duplicate_rows']} for name,p in profiles.items()}).T
display(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, profile) in zip(axes, profiles.items()):
    counts = pd.DataFrame(profile['class_distribution']).set_index('label')['count']
    counts.plot.bar(ax=ax, color='#168b7e')
    ax.set_title(name + ' outcome counts')
    ax.set_ylabel('Students')
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, profile) in zip(axes, profiles.items()):
    missing = pd.Series({c['name']: c['missing'] for c in profile['columns']})
    missing = missing[missing > 0].sort_values()
    if missing.empty:
        ax.text(.5, .5, 'No missing cells', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
    else:
        missing.plot.barh(ax=ax, color='#df7368')
    ax.set_title(name + ' missing cells')
plt.tight_layout(); plt.show()


## Interpretation
The Kaggle target uses `1` for dropout and `0` for non-dropout. The UCI target uses `Dropout`, `Enrolled`, and `Graduate`. The two non-dropout states in UCI are different outcomes and should remain visible in descriptive charts. Only create a shared binary target for a separately documented modeling experiment.
